# 11 Hybrid Lexical Semantic Investigation

**Project:** Insurance Fraud Detection Assistant

**Notebook:** `11-hybrid-lexical-semantic-investigation.ipynb`

In [ ]:
# ==========================================
# Notebook 11
# Hybrid Fraud Search
# ==========================================

import pandas as pd
import numpy as np

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
claims_df = pd.read_csv("../data/insurance_claims.csv")

In [5]:
claims_df.head()

,claim_id,claimant_name,vehicle_vin,mechanic_shop,clinic_name,lawyer,claimant_statement,police_report,adjuster_notes,medical_bill,fraud_label
0,CLM001,Wendy Holland,FqH15433919443,Rapid Auto Repair,Care First Clinic,Smith & Associates,A vehicle rear-ended me while I was waiting at...,Witnesses confirmed another driver caused the ...,Section international though many movement.,5072,0
1,CLM002,Douglas Lara,acF49501195178,Rapid Auto Repair,Wellness Recovery Center,Anderson Legal,The vehicle changed lanes unexpectedly and hit...,Police observed damage consistent with reporte...,Budget Mrs part spend middle threat smile incr...,1541,0
2,CLM003,Chloe Murphy,xeQ24677572737,Rapid Auto Repair,Care First Clinic,Justice Partners,I was stopped at a red light when another vehi...,Accident report indicates claimant followed tr...,Similar never box line.,20226,1
3,CLM004,Jodi Reynolds MD,sPL40843321198,Rapid Auto Repair,Wellness Recovery Center,Justice Partners,The vehicle changed lanes unexpectedly and hit...,Police observed damage consistent with reporte...,Section season nor political bank.,7723,0
4,CLM005,Elizabeth Patel,mmr35740163797,Prime Vehicle Repair,Wellness Recovery Center,Smith & Associates,I was driving through an intersection when ano...,Witnesses confirmed another driver caused the ...,Kind compare across audience society.,23376,0


In [6]:
documents = claims_df["claimant_statement"].astype(str).tolist()

In [7]:
len(documents)

15

In [8]:
tokenized_docs = [doc.lower().split() for doc in documents]

In [9]:
bm25 = BM25Okapi(tokenized_docs)

In [25]:
def bm25_search(query, top_k=5):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_idx:

        results.append(
            {
                "claim_id": claims_df.iloc[idx]["claim_id"],
                "score": float(scores[idx]),
                "statement": claims_df.iloc[idx]["claimant_statement"],
            }
        )

    return results

In [27]:
bm25_search("rear ended accident")

[{'claim_id': 'CLM014',
  'score': 1.815623152429359,
  'statement': 'I was traveling within the speed limit when the accident occurred.'},
 {'claim_id': 'CLM008',
  'score': 1.815623152429359,
  'statement': 'I was traveling within the speed limit when the accident occurred.'},
 {'claim_id': 'CLM015',
  'score': 0.0,
  'statement': 'I was driving through an intersection when another vehicle struck my car.'},
 {'claim_id': 'CLM013',
  'score': 0.0,
  'statement': 'A vehicle rear-ended me while I was waiting at a traffic signal.'},
 {'claim_id': 'CLM012',
  'score': 0.0,
  'statement': 'The driver failed to yield and collided with my vehicle.'}]

In [14]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [15]:
claim_embeddings = embedding_model.encode(documents, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(query)

    scores = cosine_similarity(query_embedding.reshape(1, -1), claim_embeddings)[0]

    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_idx:

        results.append(
            {
                "claim_id": claims_df.iloc[idx]["claim_id"],
                "score": float(scores[idx]),
                "statement": claims_df.iloc[idx]["claimant_statement"],
            }
        )

    return results

In [24]:
# Cell 19


def hybrid_search(query, top_k=5):

    bm25_results = bm25_search(query, top_k=20)

    semantic_results = semantic_search(query, top_k=20)

    fused_results = reciprocal_rank_fusion(bm25_results, semantic_results)

    final_results = []

    for claim_id, score in fused_results[:top_k]:

        claim = claims_df[claims_df["claim_id"] == claim_id].iloc[0]

        final_results.append(
            {
                "claim_id": claim_id,
                "rrf_score": score,
                "statement": claim["claimant_statement"],
            }
        )

    return pd.DataFrame(final_results)

In [29]:
semantic_search("car hit from behind")

[{'claim_id': 'CLM004',
  'score': 0.7054904699325562,
  'statement': 'The vehicle changed lanes unexpectedly and hit my car.'},
 {'claim_id': 'CLM002',
  'score': 0.7054904699325562,
  'statement': 'The vehicle changed lanes unexpectedly and hit my car.'},
 {'claim_id': 'CLM010',
  'score': 0.661197304725647,
  'statement': 'I was driving through an intersection when another vehicle struck my car.'},
 {'claim_id': 'CLM007',
  'score': 0.661197304725647,
  'statement': 'I was driving through an intersection when another vehicle struck my car.'},
 {'claim_id': 'CLM005',
  'score': 0.661197304725647,
  'statement': 'I was driving through an intersection when another vehicle struck my car.'}]

In [30]:
def reciprocal_rank_fusion(bm25_results, semantic_results, k=60):

    scores = {}

    for rank, result in enumerate(bm25_results, start=1):

        claim_id = result["claim_id"]

        scores.setdefault(claim_id, 0)

        scores[claim_id] += 1 / (k + rank)

    for rank, result in enumerate(semantic_results, start=1):

        claim_id = result["claim_id"]

        scores.setdefault(claim_id, 0)

        scores[claim_id] += 1 / (k + rank)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [31]:
def hybrid_search(query, top_k=5):

    bm25_results = bm25_search(query, top_k=20)

    semantic_results = semantic_search(query, top_k=20)

    fused_results = reciprocal_rank_fusion(bm25_results, semantic_results)

    final_results = []

    for claim_id, score in fused_results[:top_k]:

        claim = claims_df[claims_df["claim_id"] == claim_id].iloc[0]

        final_results.append(
            {
                "claim_id": claim_id,
                "rrf_score": score,
                "statement": claim["claimant_statement"],
            }
        )

    return pd.DataFrame(final_results)

In [32]:
hybrid_search("rear ended accident")

,claim_id,rrf_score,statement
0,CLM013,0.031498,A vehicle rear-ended me while I was waiting at...
1,CLM015,0.031258,I was driving through an intersection when ano...
2,CLM004,0.030282,The vehicle changed lanes unexpectedly and hit...
3,CLM010,0.030077,I was driving through an intersection when ano...
4,CLM014,0.029907,I was traveling within the speed limit when th...


In [33]:
hybrid_search("neck injury after collision")

,claim_id,rrf_score,statement
0,CLM011,0.032787,\n\nMy neck and lower back experienced severe ...
1,CLM006,0.032258,\n\nMy neck and lower back experienced severe ...
2,CLM015,0.031025,I was driving through an intersection when ano...
3,CLM010,0.030798,I was driving through an intersection when ano...
4,CLM007,0.029911,I was driving through an intersection when ano...


In [34]:
hybrid_search("vehicle hit from behind")

,claim_id,rrf_score,statement
0,CLM004,0.032787,The vehicle changed lanes unexpectedly and hit...
1,CLM002,0.032258,The vehicle changed lanes unexpectedly and hit...
2,CLM015,0.031498,I was driving through an intersection when ano...
3,CLM010,0.030777,I was driving through an intersection when ano...
4,CLM007,0.030310,I was driving through an intersection when ano...


In [35]:
query = """
severe neck pain after accident
"""

In [36]:
results = hybrid_search(query, top_k=10)

In [37]:
results

,claim_id,rrf_score,statement
0,CLM011,0.032787,\n\nMy neck and lower back experienced severe ...
1,CLM006,0.032258,\n\nMy neck and lower back experienced severe ...
2,CLM015,0.031258,I was driving through an intersection when ano...
3,CLM010,0.030331,I was driving through an intersection when ano...
4,CLM014,0.029958,I was traveling within the speed limit when th...
5,CLM007,0.029670,I was driving through an intersection when ano...
6,CLM008,0.029514,I was traveling within the speed limit when th...
7,CLM013,0.029437,A vehicle rear-ended me while I was waiting at...
8,CLM005,0.029236,I was driving through an intersection when ano...
9,CLM004,0.028814,The vehicle changed lanes unexpectedly and hit...


In [38]:
claim_text = claims_df.iloc[5]["claimant_statement"]

In [39]:
hybrid_search(claim_text, top_k=10)

,claim_id,rrf_score,statement
0,CLM011,0.032787,\n\nMy neck and lower back experienced severe ...
1,CLM006,0.032258,\n\nMy neck and lower back experienced severe ...
2,CLM004,0.030310,The vehicle changed lanes unexpectedly and hit...
3,CLM010,0.030159,I was driving through an intersection when ano...
4,CLM002,0.029857,The vehicle changed lanes unexpectedly and hit...
5,CLM007,0.029710,I was driving through an intersection when ano...
6,CLM015,0.029644,I was driving through an intersection when ano...
7,CLM012,0.029387,The driver failed to yield and collided with m...
8,CLM005,0.029274,I was driving through an intersection when ano...
9,CLM009,0.028958,The driver failed to yield and collided with m...


In [40]:
search_metadata = pd.DataFrame(
    {
        "claim_id": claims_df["claim_id"],
        "mechanic": claims_df["mechanic_shop"],
        "clinic": claims_df["clinic_name"],
        "lawyer": claims_df["lawyer"],
    }
)

In [41]:
search_metadata.to_csv("../data/fraud_search_metadata.csv", index=False)